# 冻结 Stage1 父本：NVDA 原始 Data Center 同期同比的 SOXX H63 信息筛选

本卡仅检验新特征是否增加中期 SOXX 价格信息。它不替代 Stage1 配置链，不产生交易账户，不构成超过 SOXX 的证据。固定方案、数据哈希、时间边界及门槛见 `plan.json`。

In [1]:
import os,sys,json,hashlib,math
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
from src.data.nas_market_data import archive_nas_recorder,materialize_nas_evidence,_settings
card=root/'research/stage1_nvda_datacenter_h63_information_20261023'
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
plan=json.loads((card/'plan.json').read_text());src=plan['sources'];assert plan['frozen'] and plan['version']==2
assert not (card/'run_state.json').exists(),'frozen once-only run already started; do not repeat source reads or fits'
assert sha(card/'plan.json')=='45f0622f009a747dce623ab19a81846bcaf74ddd2c140d1be92e9e8ad4960ea4'
qcard=root/src['numeric_source_card'];mcard=root/src['Mellanox_mask_card']
for p,h in [(qcard/'summary.json',src['numeric_summary_sha256']),(qcard/'result.json',src['numeric_result_sha256']),(mcard/'result.json',src['Mellanox_result_sha256']),(root/src['Qlib_price_provider']/'qlib.yaml',src['Qlib_config_sha256']),(root/src['Qlib_price_provider']/'features/soxx/open.day.bin',src['SOXX_open_sha256'])]:assert sha(p)==h,(p,sha(p),h)
assert json.loads((qcard/'summary.json').read_text())['nas_evidence']['sha256']==src['numeric_NAS_sha256']
ma=json.loads((mcard/'nas_source_archive.json').read_text());assert ma['verified'] and ma['sha256']==src['Mellanox_NAS_sha256'] and ma['members'][str((mcard/'result.json').relative_to(root))]==src['Mellanox_result_sha256']
source_artifact=qcard/'qlib_recorder'/src['numeric_experiment_id']/src['numeric_recorder_id']/'artifacts/analysis/source_frame'
assert sha(source_artifact)==src['numeric_frame_sha256']
mask={e['accession'] for e in json.loads((mcard/'result.json').read_text())['masked_events']};assert len(mask)==4
state={'plan_sha256':sha(card/'plan.json'),'phase':'preflight_attempted','source_recorder_load_attempts':1,'D_calendar_call_attempts':1,'D_features_call_attempts':1,'DatasetH_builds':0,'model_fit_attempts':0,'bootstrap_attempts':0,'new_Qlib_Recorders':0,'native_accounts':0}
(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
qlib.init(provider_uri=str(root/src['Qlib_price_provider']),region='us',kernels=1)
cal=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2026-10-09',freq='day'));assert cal.is_unique and cal.is_monotonic_increasing
raw=D.features(['SOXX'],['$open'],start_time='2011-01-03',end_time='2026-10-09',freq='day')
opens=raw['$open'].droplevel('instrument').reindex(cal).astype(float);assert raw.index.get_level_values('instrument').unique().tolist()==['SOXX'] and np.isfinite(opens).all() and (opens>0).all()
with R.uri_context(str(qcard/'qlib_recorder')):
  srec=R.get_recorder(recorder_id=src['numeric_recorder_id'],experiment_id=src['numeric_experiment_id'])
  assert srec.status=='FINISHED'
  source=srec.load_object('analysis/source_frame');assert source.index.equals(cal) and source.index.is_unique
source=source.copy();source.loc[source.source_accession.isin(mask),'NVDA_Data_Center_same_release_YoY']=np.nan
logp=np.log(opens);trend=logp-logp.shift(63);vol=logp.diff().rolling(20).std(ddof=1)
first=pd.Series(cal,index=cal).groupby(cal.to_period('M')).first();decisions=pd.DatetimeIndex(first.loc['2017-01':'2025-12'].to_numpy());assert len(decisions)==108
t=pd.Index(cal).get_indexer(decisions);assert (t>=63).all() and (t+64<len(cal)).all()
f=pd.DataFrame(index=decisions);f['trend63']=trend.loc[decisions].to_numpy();f['vol20']=vol.loc[decisions].to_numpy();f['source_age']=source.loc[decisions,'source_age_calendar_days'].to_numpy();f['dc_yoy']=source.loc[decisions,'NVDA_Data_Center_same_release_YoY'].to_numpy();f['source_accession']=source.loc[decisions,'source_accession'].to_numpy();f['maturity']=cal[t+64]
f['eligible']=np.isfinite(f[['trend63','vol20','source_age','dc_yoy']]).all(axis=1)
train=f.loc['2017-01':'2022-12'].query('eligible').copy();train=train.loc[train.maturity<pd.Timestamp('2023-01-03')]
held=f.loc['2023-01':'2025-12'].query('eligible').copy()
def support(z):
  a=z.source_accession.dropna();ages=z.source_age.astype(float);pos=pd.Index(cal).get_indexer(z.index);starts=pos+1;ends=pos+64
  carry={}
  if len(z):
    daily=source.loc[z.index.min():z.index.max(),'source_accession']
    for accession in a.unique():
      days=daily.index[daily.eq(accession)]
      if len(days):carry[str(accession)]=int((days.max()-days.min()).days+1)
  return {'monthly_rows':len(z),'unique_original_accessions':int(a.nunique()),'source_carry_calendar_days_by_accession':carry,'source_age_min_days':float(ages.min()) if len(z) else None,'source_age_median_days':float(ages.median()) if len(z) else None,'source_age_max_days':float(ages.max()) if len(z) else None,'adjacent_H63_label_overlaps':int(sum(starts[i]<=ends[i-1] for i in range(1,len(z))))}
halves=[held.loc['2023-01-03':'2024-06-30'],held.loc['2024-07-01':'2025-12-31']];years={str(y):held.loc[str(y)] for y in (2023,2024,2025)}
preflight={'plan_sha256':sha(card/'plan.json'),'source_frame_sha256':sha(source_artifact),'train':support(train),'held':support(held),'halves':[support(x) for x in halves],'held_years':{k:support(v) for k,v in years.items()},'train_years':{str(y):support(train.loc[str(y)]) for y in range(2017,2023)},'train_monthly_decisions_all':72,'held_monthly_decisions_all':36,'2026_decisions':0,'outcome_values_read':0,'model_fits':0}
assert len(train)>=48 and len(held)==36 and train.source_accession.nunique()>=16 and held.source_accession.nunique()>=10
assert [len(x) for x in halves]==[18,18] and [len(x) for x in years.values()]==[12,12,12]
assert (train.maturity<pd.Timestamp('2023-01-03')).all() and held.index.max().year==2025
(card/'preflight.json').write_text(json.dumps(preflight,indent=2)+'\n')
state['phase']='preflight_complete';(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');print(preflight)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[33018:MainThread](2026-10-23 08:34:35,013) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[33018:MainThread](2026-10-23 08:34:35,015) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[33018:MainThread](2026-10-23 08:34:35,015) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


{'plan_sha256': '45f0622f009a747dce623ab19a81846bcaf74ddd2c140d1be92e9e8ad4960ea4', 'source_frame_sha256': '9c44622d04024673ce487671d4866b78499c83c2ed2528898e0a45d16b94c7d5', 'train': {'monthly_rows': 57, 'unique_original_accessions': 20, 'source_carry_calendar_days_by_accession': {'0001045810-16-000343': 38, '0001045810-17-000016': 89, '0001045810-17-000068': 93, '0001045810-17-000117': 91, '0001045810-17-000164': 91, '0001045810-18-000004': 91, '0001045810-18-000052': 98, '0001045810-18-000113': 91, '0001045810-18-000148': 91, '0001045810-19-000007': 91, '0001045810-19-000077': 91, '0001045810-19-000140': 91, '0001045810-19-000168': 91, '0001045810-20-000007': 98, '0001045810-20-000063': 90, '0001045810-21-000128': 91, '0001045810-21-000160': 91, '0001045810-22-000008': 98, '0001045810-22-000073': 91, '0001045810-22-000136': 8}, 'source_age_min_days': 6.0, 'source_age_median_days': 47.0, 'source_age_max_days': 83.0, 'adjacent_H63_label_overlaps': 55}, 'held': {'monthly_rows': 36, 'un

## 拟合与锁定的 held 检验
仅当前一单元全部通过才运行。两个 Ridge 用完全匹配的训练日期；结果不能调参补救。

In [2]:
# Qlib DatasetH 保存完整月度特征及训练标签；held 标签保持未知直到两个模型已拟合。
assert sha(card/'plan.json')=='45f0622f009a747dce623ab19a81846bcaf74ddd2c140d1be92e9e8ad4960ea4'
assert json.loads((card/'preflight.json').read_text())==preflight and preflight['plan_sha256']==sha(card/'plan.json')
for p,h in [(qcard/'summary.json',src['numeric_summary_sha256']),(qcard/'result.json',src['numeric_result_sha256']),(mcard/'result.json',src['Mellanox_result_sha256']),(root/src['Qlib_price_provider']/'qlib.yaml',src['Qlib_config_sha256']),(root/src['Qlib_price_provider']/'features/soxx/open.day.bin',src['SOXX_open_sha256']),(source_artifact,src['numeric_frame_sha256'])]:assert sha(p)==h
assert json.loads((qcard/'summary.json').read_text())['nas_evidence']['sha256']==src['numeric_NAS_sha256']
assert json.loads((mcard/'nas_source_archive.json').read_text())['sha256']==src['Mellanox_NAS_sha256']
state=json.loads((card/'run_state.json').read_text());assert state['phase']=='preflight_complete' and state['plan_sha256']==sha(card/'plan.json')
state['phase']='pre_fit';(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
label=pd.Series(np.nan,index=f.index,name='h63_log_return');label.loc[train.index]=(logp.iloc[pd.Index(cal).get_indexer(train.index)+64].to_numpy()-logp.iloc[pd.Index(cal).get_indexer(train.index)+1].to_numpy())
feature_cols=['trend63','vol20','source_age','dc_yoy'];panel=pd.concat({'feature':f[feature_cols],'label':label.to_frame()},axis=1);panel.index=pd.MultiIndex.from_arrays([f.index,['SOXX']*len(f)],names=['datetime','instrument'])
ds=DatasetH(handler=DataHandlerLP.from_df(panel),segments={'train':('2017-01-01','2022-12-31'),'held':('2023-01-03','2025-12-31')})
state=json.loads((card/'run_state.json').read_text());state['DatasetH_builds']=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
prepared=ds.prepare('train',col_set=['feature','label']);assert train.index.equals(pd.DatetimeIndex(prepared.index.get_level_values('datetime')[prepared['label']['h63_log_return'].notna()]))
xtr=prepared.loc[pd.IndexSlice[train.index,:],:];ytr=xtr['label']['h63_log_return'].to_numpy(float)
assert np.isfinite(ytr).all() and len(ytr)==len(train)
base_cols=feature_cols[:3];base_scaler=StandardScaler().fit(xtr['feature'][base_cols]);cand_scaler=StandardScaler().fit(xtr['feature'][feature_cols])
state['model_fit_attempts']=2;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
base=Ridge(alpha=10.0,fit_intercept=True).fit(base_scaler.transform(xtr['feature'][base_cols]),ytr)
candidate=Ridge(alpha=10.0,fit_intercept=True).fit(cand_scaler.transform(xtr['feature'][feature_cols]),ytr)
fit_lock={'plan_sha256':sha(card/'plan.json'),'train_rows':len(train),'train_label_sha256':hashlib.sha256(ytr.tobytes()).hexdigest(),'base_coefficients':base.coef_.tolist(),'candidate_coefficients':candidate.coef_.tolist(),'added_standardized_coefficient':float(candidate.coef_[-1]),'fits':2}
(card/'fit_lock.json').write_text(json.dumps(fit_lock,indent=2)+'\n')
assert ds.prepare('held',col_set='label')['h63_log_return'].isna().all()
xh=f.loc[held.index,feature_cols];pb=base.predict(base_scaler.transform(xh[base_cols]));pc=candidate.predict(cand_scaler.transform(xh[feature_cols]))
hp=pd.Index(cal).get_indexer(held.index);yh=(logp.iloc[hp+64].to_numpy()-logp.iloc[hp+1].to_numpy())
rows=held[['source_accession','source_age','maturity']].copy();rows['target']=yh;rows['baseline_prediction']=pb;rows['candidate_prediction']=pc;rows['baseline_squared_error']=(yh-pb)**2;rows['candidate_squared_error']=(yh-pc)**2;rows['paired_MSE_improvement']=rows.baseline_squared_error-rows.candidate_squared_error
state['bootstrap_attempts']=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
d=rows.paired_MSE_improvement.to_numpy();rng=np.random.default_rng(20261023);boot=np.empty(10000);n=len(d)
for b in range(10000):
  ix=np.empty(n,dtype=int);ix[0]=rng.integers(n)
  for j in range(1,n):ix[j]=rng.integers(n) if rng.random()<1/3 else (ix[j-1]+1)%n
  boot[b]=d[ix].mean()
lower=float(np.quantile(boot,0.025/15));full=float(d.mean());half_means=[float(rows.loc[z.index,'paired_MSE_improvement'].mean()) for z in halves];year_means={k:float(rows.loc[z.index,'paired_MSE_improvement'].mean()) for k,z in years.items()}
passed=bool(candidate.coef_[-1]>0 and full>0 and lower>0 and all(x>0 for x in half_means) and all(x>0 for x in year_means.values()))
result={'decision':'information_gate_passed_only' if passed else 'not_promoted_information_gate_failed','plan_sha256':sha(card/'plan.json'),'preflight':preflight,'fit_lock':fit_lock,'held_rows':len(rows),'held_mean_paired_MSE_improvement':full,'held_half_means':half_means,'held_year_means':year_means,'stationary_bootstrap_lower_0p025_over_15':lower,'gate_passed':passed,'native_accounts':0,'source_first_public_timing_qualified':False,'platform_definition_continuity_qualified':False,'independent_validation':False}
state['new_Qlib_Recorders']=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
  rec=R.get_recorder();rid=rec.id;eid=str(rec.experiment_id);ds.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,dataset=ds,preflight=preflight);rec.save_objects(artifact_path='models',baseline=base,candidate=candidate,baseline_scaler=base_scaler,candidate_scaler=cand_scaler,fit_lock=fit_lock);rec.save_objects(artifact_path='predictions',monthly_rows=rows);rec.save_objects(artifact_path='analysis',result=result,bootstrap_means=boot);rec.log_metrics(held_mean_MSE_improvement=full,bootstrap_lower=lower,added_coefficient=float(candidate.coef_[-1]),gate_passed=int(passed),model_fits=2,native_accounts=0)
result['recorder_id']=rid;result['experiment_id']=eid;(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'result':'result.json','model_fits':2,'native_accounts':0,'artifact_policy':plan['artifact_policy']}
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
summary['nas_evidence']=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
import tempfile
with tempfile.TemporaryDirectory(prefix='h63-nas-verify-') as tmp:
  restored=materialize_nas_evidence(binding,card=Path(tmp));assert restored['recorder_id']==rid and restored['archive_sha256']==summary['nas_evidence']['sha256']
  with R.uri_context(str(Path(tmp)/'qlib_recorder')):
    check=R.get_recorder(recorder_id=rid,experiment_id=eid);assert check.status=='FINISHED' and check.load_object('analysis/result')['gate_passed']==passed
summary['nas_readback']={'verified':True,'sha256':summary['nas_evidence']['sha256']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['phase']='finished';state['recorder_id']=rid;state['nas_sha256']=summary['nas_evidence']['sha256'];(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print(result['decision'],full,half_means,year_means,lower,rid)


[33018:MainThread](2026-10-23 08:34:35,088) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[33018:MainThread](2026-10-23 08:34:35,088) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[33018:MainThread](2026-10-23 08:34:35,088) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[33018:MainThread](2026-10-23 08:34:35,284) WARNING - qlib.workflow - [expm.py:230] - No valid experiment found. Create a new experiment with name stage1_nvda_datacenter_h63_information_20261023.


[33018:MainThread](2026-10-23 08:34:35,286) INFO - qlib.workflow - [exp.py:258] - Experiment 669660884845739876 starts running ...


[33018:MainThread](2026-10-23 08:34:35,341) INFO - qlib.workflow - [recorder.py:345] - Recorder fa66afadeffc4ac098b176c52ed10095 starts running under Experiment 669660884845739876 ...


[33018:MainThread](2026-10-23 08:34:36,124) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


not_promoted_information_gate_failed -0.0001303412379792872 [0.0006436319849090379, -0.0009043144608676118] {'2023': 0.001429722688761518, '2024': -0.0016398551759311739, '2025': -0.0001808912267682049} -0.0032491154969649643 fa66afadeffc4ac098b176c52ed10095


## 中文结论

固定 v2 方案（`plan.json` SHA-256 `45f0622f…`）一次执行完成，原生 Qlib Recorder `fa66afadeffc4ac098b176c52ed10095` 为 FINISHED，NAS 归档 SHA-256 `0605e338…` 已限定恢复并原生读回。训练 57 个按月决策、20 个原始 SEC accession；2023–2025 留出 36 个月、13 个 accession；两半各 18 月。留出相邻标签 35 对重叠，同一披露在多个月份沿用，独立事件数远少于月数。

新增 Data Center 同期同比的训练标准化系数为 +0.00956，但相对三控制项的留出平均配对 MSE 改善为 -0.000130；前 18 个月 +0.000644，后 18 个月 -0.000904；2023 年 +0.001430、2024 年 -0.001640、2025 年 -0.000181。预注册平稳区块 bootstrap 下界 -0.003249。全段、后半段、2024、2025 及区块下界均未过门槛，因此固定 Data Center YoY/H63/Ridge 版本 **not_promoted**，不进入 Stage1 原 39 特征接收器或费用账户，也不能声称超过 SOXX。

本次只检验价差预测信息，没有 SOXL/SOXX/CASH 交易或费用结果。SEC 接收后下一 Qlib 日是保守可用时钟，发行人首次公开时刻和平台业务口径连续性仍未认证；历史数据也不是独立前向证据。下一步在冻结 Stage1 的 0bp 全账户差区间中，对比 2023 好段与 2024–2025 差段的现金机会成本、SOXL 相对选择和风险反馈，重新冻结实质不同的可检验特征机制；不得改本卡阈值、符号或窗口。
